# Fracture Classifier — Kaggle Training Notebook

Runs the full pipeline: split → train → evaluate → upload to Hugging Face.

## Checklist vor dem Start
- [ ] Accelerator: **GPU T4 x2** (Settings → Accelerator)
- [ ] Internet: **On** (Settings → Internet)
- [ ] Bilddatensatz hinzugefügt (Add Data → `venthanforearm-xrays`)
- [ ] Hugging Face **Write Token** bereit (huggingface.co → Settings → Access Tokens → New → Classic → Write)

## Ablauf
| Cell | Was passiert |
|---|---|
| 1 | Pakete installieren (einmalig, Kernel-Neustart) |
| 2 | Konfiguration + GPU-Check |
| 3 | Code von GitHub klonen |
| 4 | Verzeichnisse + Splits CSV erstellen |
| 5 | Training (~2–5h) |
| 6 | Evaluation + eval_results.json |
| 7 | Plots anzeigen |
| 8 | Modell + JSON zu Hugging Face hochladen |

## Modell-Optionen
| Model | IMG_SIZE | BATCH_SIZE | VRAM | ~min/epoch |
|---|---|---|---|---|
| `efficientnet_b0` | 224 | 32 | ~4 GB | 1–2 min |
| `tf_efficientnetv2_m` | 480 | 16 | ~10 GB | 4–6 min |
| `tf_efficientnetv2_m` | 384 | 24 | ~8 GB | 3–4 min |

In [ ]:
# Cell 1 — Pakete installieren
# Beim ersten Mal: installiert timm + huggingface_hub und startet Kernel neu.
# Beim zweiten Mal: bereits installiert → kein Neustart.
try:
    import timm
    from huggingface_hub import HfApi
    print(f"timm {timm.__version__} — bereits installiert, kein Neustart nötig")
except ImportError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "timm", "huggingface_hub", "-q"],
        check=True
    )
    print("Installiert. Kernel wird neu gestartet — danach ab Cell 2 weiterlaufen lassen.")
    import IPython
    IPython.get_ipython().kernel.do_shutdown(restart=True)

In [ ]:
# Cell 2 — Konfiguration + GPU-Check
import os
from pathlib import Path

# ══════════════════════════════════════════════════════════════════════
# NUR DIESE ZEILEN ANPASSEN
IMAGE_DATASET_SLUG = "venthanforearm-xrays"       # Kaggle Bild-Dataset Slug
HF_TOKEN           = "hf_DEIN_WRITE_TOKEN"        # huggingface.co → Settings → Access Tokens
HF_REPO_ID         = "VenthanVi/fracture-detection"  # wird auto-erstellt falls nicht vorhanden
# ══════════════════════════════════════════════════════════════════════

# ── DATA_ROOT auto-detection ──────────────────────────────────────────
_candidates = [Path(f"/kaggle/input/{IMAGE_DATASET_SLUG}/alle Bilder")]
_datasets_dir = Path("/kaggle/input/datasets")
if _datasets_dir.exists():
    _candidates += [p / "alle Bilder" for p in _datasets_dir.glob(f"*/{IMAGE_DATASET_SLUG}")]

_data_root = next((p for p in _candidates if p.exists()), None)
if _data_root is None:
    hits = list(Path("/kaggle/input").rglob("alle Bilder"))
    _data_root = hits[0] if hits else None

if _data_root is None:
    raise FileNotFoundError(
        f"'alle Bilder' nicht gefunden.\n"
        f"Verfügbare Inputs: {os.listdir('/kaggle/input/')}\n"
        f"IMAGE_DATASET_SLUG = '{IMAGE_DATASET_SLUG}'"
    )

os.environ["DATA_ROOT"]      = str(_data_root)
os.environ["SPLITS_CSV"]     = "/kaggle/working/data/splits.csv"
os.environ["CHECKPOINT_DIR"] = "/kaggle/working/checkpoints"
os.environ["LOG_DIR"]        = "/kaggle/working/logs"
os.environ["MODEL_NAME"]     = "tf_efficientnetv2_m"
os.environ["IMG_SIZE"]       = "480"
os.environ["BATCH_SIZE"]     = "16"
os.environ["NUM_WORKERS"]    = "4"

# ── GPU-Check ─────────────────────────────────────────────────────────
import torch
assert torch.cuda.is_available(), "Kein GPU — Accelerator in Settings aktivieren!"
print(f"GPU  : {torch.cuda.get_device_name(0)}")
print(f"VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"Daten: {os.environ['DATA_ROOT']}")
print(f"Model: {os.environ['MODEL_NAME']}  {os.environ['IMG_SIZE']}px  batch={os.environ['BATCH_SIZE']}")

In [ ]:
# Cell 3 — Code von GitHub klonen (immer aktuellste Version)
import sys, subprocess
from pathlib import Path

REPO_URL  = "https://github.com/VenthanV/FractureDataset.git"
CODE_PATH = Path("/kaggle/working/fracture")

if CODE_PATH.exists():
    # Bereits geklont → auf neusten Stand bringen
    result = subprocess.run(
        ["git", "-C", str(CODE_PATH), "pull"],
        capture_output=True, text=True
    )
    print(f"git pull: {result.stdout.strip() or result.stderr.strip()}")
else:
    result = subprocess.run(
        ["git", "clone", "--depth=1", REPO_URL, str(CODE_PATH)],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError(
            "git clone fehlgeschlagen!\n"
            "1. Settings → Internet → ON\n"
            "2. GitHub-Repo muss PUBLIC sein"
        )
    print("Geklont.")

sys.path.insert(0, str(CODE_PATH))
print(f"Code-Pfad: {CODE_PATH}")

In [ ]:
# Cell 4 — Create output directories and build train/val/test split CSV
from pathlib import Path
Path("/kaggle/working/data").mkdir(parents=True, exist_ok=True)
Path("/kaggle/working/checkpoints").mkdir(parents=True, exist_ok=True)
Path("/kaggle/working/logs").mkdir(parents=True, exist_ok=True)

import dataset
dataset.main()

In [ ]:
# Cell 5 — Train (Phase 1: frozen backbone → Phase 2: fine-tune last 2 blocks)
# Expected runtime on T4 at 480px:
#   Phase 1 (20 epochs × ~5 min) ≈ 100 min
#   Phase 2 (up to 40 epochs × ~5 min, early stop) ≈ 60–200 min
#   Total ≈ 2.5–5 hours  (well within Kaggle's 9-hour session limit)
import train
train.main()

In [ ]:
# Cell 6 — Evaluation + eval_results.json speichern
import sys, importlib

# Jupyter übergibt eigene Args an sys.argv → vor argparse bereinigen
sys.argv = [sys.argv[0], "--save-json"]

import evaluate
importlib.reload(evaluate)  # sicherstellen dass die neueste Version geladen ist
evaluate.main()

In [ ]:
# Cell 7 — Display result plots inline
from IPython.display import Image, display
from pathlib import Path

log_dir = Path("/kaggle/working/logs")
for plot in ["roc_curve.png", "confusion_matrix.png"]:
    path = log_dir / plot
    if path.exists():
        print(f"\n{plot}:")
        display(Image(filename=str(path)))

In [ ]:
# Cell 8 — Modell + eval_results.json zu Hugging Face Hub hochladen
# HF_TOKEN und HF_REPO_ID wurden in Cell 2 gesetzt
from huggingface_hub import HfApi, login
from pathlib import Path

login(token=HF_TOKEN)
api = HfApi()
api.create_repo(HF_REPO_ID, repo_type="model", exist_ok=True)

uploads = {
    "/kaggle/working/checkpoints/best_model.pth": "best_model.pth",
    "/kaggle/working/logs/eval_results.json":     "eval_results.json",
}

for local, remote in uploads.items():
    if Path(local).exists():
        api.upload_file(
            path_or_fileobj=local,
            path_in_repo=remote,
            repo_id=HF_REPO_ID,
        )
        print(f"Hochgeladen: {remote}")
    else:
        print(f"Nicht gefunden: {local}")

print(f"\nFertig: https://huggingface.co/{HF_REPO_ID}")